<a href="https://colab.research.google.com/github/REKNOS/SQL-Database/blob/main/real_estate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Import required libraries numpy and pandas
# sqlite3 - used to create and manage SQLite database

import numpy as np
import pandas as pd
import sqlite3

In [5]:
#Fixing random seed ensures deterministic output.
np.random.seed(42)
#Define dataset sizes
n_properties = 1000
n_agents = 50
n_listings = 2000

In [6]:
# Generates Agent Table data
# First and last name pools for realistic name generation
first_names = [
    "Rio", "Stephan", "Oliver", "Noah", "Liam", "Mason", "Lucas", "Ethan",
    "Ava", "Sophia", "Isabella", "Mia", "Charlotte", "Amelia", "Harper",
    "James", "Benjamin", "Elijah", "William", "Henry"
]

last_names = [
    "Masoor", "Smith", "Johnson", "Brown", "Taylor", "Anderson",
    "Thomas", "Jackson", "White", "Harris", "Martin", "Thompson",
    "Garcia", "Clark", "Lewis", "Walker", "Hall", "Allen"
]
# unique agents Id
agents_id = np.arange(1, n_agents + 1)

# Random Full names
random_names = [
    f"{np.random.choice(first_names)} {np.random.choice(last_names)}"
    for _ in range(n_agents)
]

regions = np.random.choice(['North', 'South', 'East', 'West'], n_agents)
ages = np.random.randint(22, 65, n_agents)#realistic age ranges

# Generate synthetic emails
emails = [
    name.lower().replace(" ", ".") + str(agent_id) + "@realestate.com"
    for name, agent_id in zip(random_names, agents_id)
]
# UK style phone numbers
phone_numbers = [
    f"07{np.random.randint(100000000, 999999999)}"
    for _ in range(n_agents)
]
# Create agents data frame using pandas
agents_df = pd.DataFrame({
    'agents_ID': agents_id,
    'agent_name': random_names,
    'age': ages,
    'email': emails,
    'phone_number': phone_numbers,
    'region': regions
})

# Introduce NULL values (10%)
email_missing = np.random.rand(n_agents) < 0.10
phone_missing = np.random.rand(n_agents) < 0.10

agents_df.loc[email_missing, 'email'] = None
agents_df.loc[phone_missing, 'phone_number'] = None

agents_df.head()

,agents_ID,agent_name,age,email,phone_number,region
0,1,Lucas Lewis,27,lucas.lewis1@realestate.com,07447262390,North
1,2,Isabella Jackson,49,isabella.jackson2@realestate.com,07149905866,East
2,3,Lucas Martin,49,lucas.martin3@realestate.com,None,North
3,4,Isabella Brown,41,isabella.brown4@realestate.com,07328233509,East
4,5,Ethan Johnson,51,ethan.johnson5@realestate.com,07605874711,South


In [7]:
#Generate PROPERTIES table data
properties_id = np.arange(1, n_properties + 1)

properties_df = pd.DataFrame({
    'property_id': properties_id,
    'property_type': np.random.choice(['Apartment', 'House', 'Villa', 'Studio'], n_properties),
    'feature_name': np.random.choice(['Garage', 'Garden', 'Balcony', 'Pool', 'Fireplace'], n_properties),
    'city': np.random.choice(['London', 'Edinburgh', 'Chester', 'Glasgow', 'Cardiff'], n_properties),
    'bedrooms': np.random.randint(1, 6, n_properties),
    'area_sqft': np.random.randint(400, 3000, n_properties),
    'year_built': np.random.randint(1960, 2024, n_properties),
    'condition_rating': np.random.randint(1, 6, n_properties)
})

# Introduce 5% NULL values
missing_mask = np.random.rand(len(properties_df)) < 0.05
properties_df.loc[missing_mask, 'condition_rating'] = None

properties_df.head()


,property_id,property_type,feature_name,city,bedrooms,area_sqft,year_built,condition_rating
0,1,House,Garden,Cardiff,5,2956,1984,5.0
1,2,Villa,Balcony,London,5,789,1969,4.0
2,3,House,Pool,Chester,1,694,1973,1.0
3,4,Apartment,Garden,Glasgow,1,1792,2001,1.0
4,5,House,Fireplace,Edinburgh,1,2117,2016,3.0


In [8]:
# Generate LISTINGS table data
# This table connects agents and properties table
# Random listing dates between 2019 and 2024
listing_dates = pd.to_datetime(
    np.random.choice(
        pd.date_range("2019-01-01", "2024-12-31"),
        n_listings
    )
).astype(str)

listings_df = pd.DataFrame({
    'property_id': np.random.choice(properties_id, n_listings),
    'agents_id': np.random.choice(agents_id, n_listings),
    'listing_dates': listing_dates,
    'price': np.round(np.random.uniform(80000, 1200000, n_listings), 2),
    'days_on_market': np.random.randint(1, 365, n_listings)
})

# Introduce deliberate duplicates (2%)
duplicate_rows = listings_df.sample(frac=0.02, random_state=42)
listings_df = pd.concat([listings_df, duplicate_rows], ignore_index=True)

listings_df.head()

,property_id,agents_id,listing_dates,price,days_on_market
0,959,44,2024-03-22,364723.97,262
1,833,47,2023-06-02,1045534.18,216
2,820,2,2024-12-06,1080656.68,312
3,747,30,2019-12-01,297451.02,49
4,986,30,2022-09-21,226110.31,92


In [9]:
#Create SQLite Database
conn = sqlite3.connect("real_estate.db")
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")# Enable foreign key enforcement

# Drop tables if they already exist
cursor.execute("DROP TABLE IF EXISTS listings;")
cursor.execute("DROP TABLE IF EXISTS properties;")
cursor.execute("DROP TABLE IF EXISTS agents;")


In [10]:
# Create SQLite database and tables
# Includes primary keys, foreign keys and composite key
# Create AGENTS table with constraints
cursor.execute("""
CREATE TABLE agents (
    agents_id INTEGER PRIMARY KEY,
    agent_name TEXT NOT NULL,
    age INTEGER CHECK(age BETWEEN 18 AND 70),
    email TEXT UNIQUE,
    phone_number TEXT,
    region TEXT
);
""")
# Create PROPERTIES table with constraints
cursor.execute("""
CREATE TABLE properties (
    property_id INTEGER PRIMARY KEY,
    property_type TEXT,
    feature_name TEXT,
    city TEXT,
    bedrooms INTEGER CHECK(bedrooms > 0),
    area_sqft INTEGER CHECK(area_sqft > 0),
    year_built INTEGER,
    condition_rating INTEGER CHECK(condition_rating BETWEEN 1 AND 5)
);
""")
# Create LISTINGS table with composite primary key
cursor.execute("""
CREATE TABLE listings (
    property_id INTEGER,
    agents_id INTEGER,
    listing_dates TEXT,
    price REAL CHECK(price > 0),
    days_on_market INTEGER CHECK(days_on_market >= 0),
    PRIMARY KEY (property_id, agents_id, listing_dates),
    FOREIGN KEY (property_id) REFERENCES properties(property_id),
    FOREIGN KEY (agents_id) REFERENCES agents(agents_id)
);
""")

conn.commit()

In [11]:
#Insert Data into Tables
listings_df.drop_duplicates(
    subset=['property_id', 'agents_id', 'listing_dates'],
    inplace=True
)
agents_df.to_sql("agents", conn, if_exists="append", index=False)
properties_df.to_sql("properties", conn, if_exists="append", index=False)
listings_df.to_sql("listings", conn, if_exists="append", index=False)

conn.commit()
conn.close()

print("Database Created Successfully")

Database Created Successfully
